In [ ]:
# ============================================================================
# VERIFICAÇÃO: Hipótese Cluster → Rápido / Disperso → Lento
# ============================================================================
# Hipótese ecológica:
#   Áreas CC* (cluster de conversão) têm mais conversão rápida (label=1)
#   Áreas CD* (conversão dispersa) têm mais conversão lenta (label=0)
#
# Se confirmada: routing determinístico por padrão é justificado
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats

BASE_DIR = Path(r"D:\Projetos\Cerrado\GeoFM_sampling")
OUT_DIR  = BASE_DIR / "dataset_v2"

# Carregar dataset v2 completo
df = pd.read_csv(OUT_DIR / "dataset_v2_full.csv")
df = df[df['label'].notna()].copy()
df['label'] = df['label'].astype(int)

print(f"Dataset v2: {len(df):,} pixels")
print(f"Colunas: {df.columns.tolist()}")
print(f"\nLabel=1 (rápido): {(df['label']==1).sum():,} ({100*(df['label']==1).mean():.1f}%)")
print(f"Label=0 (lento):  {(df['label']==0).sum():,} ({100*(df['label']==0).mean():.1f}%)")

In [ ]:
# ============================================================================
# ANÁLISE 1 — PROPORÇÃO DE CONVERSÃO RÁPIDA POR PADRÃO (Class8590)
# ============================================================================

# Classificar padrões em grupos
def classificar_padrao(p):
    if str(p).startswith('CC'): return 'Cluster-Cluster (CC*)'
    if str(p).startswith('CD'): return 'Cluster-Disperso (CD*)'
    if str(p) == 'NC':          return 'Nao Convertido (NC)'
    return 'Outro'

df['grupo'] = df['Class8590'].apply(classificar_padrao)

# Proporção de label=1 por padrão individual
stats_padrao = df.groupby('Class8590').agg(
    n          = ('label', 'count'),
    n_rapido   = ('label', 'sum'),
    pct_rapido = ('label', 'mean'),
    grupo      = ('grupo', 'first')
).reset_index().sort_values('pct_rapido', ascending=False)

print("Proporção de conversão rápida (label=1) por padrão:")
print(f"\n{'Padrão':<8} {'Grupo':<25} {'n':>6} {'n_rápido':>9} {'% rápido':>10}")
print("-" * 63)
for _, r in stats_padrao.iterrows():
    bar = '█' * int(r['pct_rapido'] * 20)
    print(f"{r['Class8590']:<8} {r['grupo']:<25} {r['n']:>6} "
          f"{r['n_rapido']:>9} {100*r['pct_rapido']:>9.1f}%  {bar}")

# Por grupo agregado
print(f"\n\nPor grupo agregado:")
stats_grupo = df.groupby('grupo').agg(
    n          = ('label', 'count'),
    pct_rapido = ('label', 'mean')
).reset_index().sort_values('pct_rapido', ascending=False)

print(f"\n{'Grupo':<25} {'n':>6} {'% rápido':>10}")
print("-" * 45)
for _, r in stats_grupo.iterrows():
    print(f"{r['grupo']:<25} {r['n']:>6} {100*r['pct_rapido']:>9.1f}%")

In [ ]:
# ============================================================================
# ANÁLISE 2 — TESTE ESTATÍSTICO: CC* vs CD*
# ============================================================================

cc_labels = df[df['Class8590'].str.startswith('CC')]['label'].values
cd_labels = df[df['Class8590'].str.startswith('CD')]['label'].values
nc_labels = df[df['Class8590'] == 'NC']['label'].values

print("Teste estatístico — proporção de conversão rápida:")
print()

pct_cc = cc_labels.mean() * 100
pct_cd = cd_labels.mean() * 100
pct_nc = nc_labels.mean() * 100 if len(nc_labels) > 0 else None

print(f"  CC* (cluster):   {pct_cc:.1f}%  (n={len(cc_labels):,})")
print(f"  CD* (disperso):  {pct_cd:.1f}%  (n={len(cd_labels):,})")
if pct_nc is not None:
    print(f"  NC:              {pct_nc:.1f}%  (n={len(nc_labels):,})")

# Teste qui-quadrado de independência
contingency = pd.crosstab(df['grupo'], df['label'])
chi2, p_chi, dof, _ = stats.chi2_contingency(contingency)
print(f"\nTeste qui-quadrado (padrão × label):")
print(f"  chi2={chi2:.2f}, df={dof}, p={p_chi:.6f}")
print(f"  {'✅ Associação significativa' if p_chi < 0.05 else '⚠️  Sem associação significativa'}")

# Teste específico CC* vs CD*
obs = np.array([
    [cc_labels.sum(), len(cc_labels) - cc_labels.sum()],
    [cd_labels.sum(), len(cd_labels) - cd_labels.sum()]
])
chi2_cc_cd, p_cc_cd, _, _ = stats.chi2_contingency(obs)
print(f"\nTeste CC* vs CD*:")
print(f"  chi2={chi2_cc_cd:.2f}, p={p_cc_cd:.6f}")
print(f"  Diferença: {pct_cc - pct_cd:+.1f}pp")
print(f"  {'✅ Significativo' if p_cc_cd < 0.05 else '⚠️  Não significativo'}")

# Effect size (Cohen's h para proporções)
h = 2 * (np.arcsin(np.sqrt(cc_labels.mean())) -
          np.arcsin(np.sqrt(cd_labels.mean())))
magnitude = 'pequeno' if abs(h) < 0.2 else 'médio' if abs(h) < 0.5 else 'grande'
print(f"  Effect size (Cohen's h): {h:.3f} ({magnitude})")

In [ ]:
# ============================================================================
# ANÁLISE 3 — DISTRIBUIÇÃO DE T POR PADRÃO E LABEL
# ============================================================================
# Pixels CC* com T recente têm conversão rápida?
# Pixels CD* com T antigo têm conversão lenta?

print("Distribuição de T por grupo e label:")
print()
for grupo in ['Cluster-Cluster (CC*)', 'Cluster-Disperso (CD*)', 'Nao Convertido (NC)']:
    sub = df[df['grupo'] == grupo]
    if len(sub) == 0: continue
    r0 = sub[sub['label']==0]['T']
    r1 = sub[sub['label']==1]['T']
    print(f"  {grupo} (n={len(sub):,}):")
    if len(r1) > 0:
        print(f"    Label=1 (rápido): T mediana={r1.median():.0f}, "
              f"range=[{r1.min():.0f},{r1.max():.0f}]")
    if len(r0) > 0:
        print(f"    Label=0 (lento):  T mediana={r0.median():.0f}, "
              f"range=[{r0.min():.0f},{r0.max():.0f}]")
    print()

In [ ]:
# ============================================================================
# GRÁFICO COMPLETO + VEREDICTO
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Proporção de rápido por padrão
cores = ['#c0392b' if p.startswith('CC') else
         '#2980b9' if p.startswith('CD') else
         '#7f8c8d' for p in stats_padrao['Class8590']]
axes[0,0].barh(stats_padrao['Class8590'],
               stats_padrao['pct_rapido']*100, color=cores)
axes[0,0].axvline(x=df['label'].mean()*100, color='black',
                  linestyle='--', label=f'Média geral ({df["label"].mean()*100:.1f}%)')
axes[0,0].set_xlabel('% conversão rápida (label=1)')
axes[0,0].set_title('Conversão rápida por padrão Class8590', fontweight='bold')
axes[0,0].legend()
# Legenda de cores
from matplotlib.patches import Patch
axes[0,0].legend(handles=[
    Patch(color='#c0392b', label='CC* (cluster)'),
    Patch(color='#2980b9', label='CD* (disperso)'),
    Patch(color='#7f8c8d', label='NC'),
], loc='lower right')

# 2. Proporção por grupo agregado
cores_g = ['#c0392b' if 'CC' in g else '#2980b9' if 'CD' in g
           else '#7f8c8d' for g in stats_grupo['grupo']]
axes[0,1].bar(range(len(stats_grupo)), stats_grupo['pct_rapido']*100,
              color=cores_g)
axes[0,1].set_xticks(range(len(stats_grupo)))
axes[0,1].set_xticklabels(
    [g.replace(' (', '\n(') for g in stats_grupo['grupo']],
    fontsize=9)
axes[0,1].axhline(y=df['label'].mean()*100, color='black',
                  linestyle='--', alpha=0.5)
axes[0,1].set_ylabel('% conversão rápida')
axes[0,1].set_title('Por grupo agregado', fontweight='bold')
for i, (_, r) in enumerate(stats_grupo.iterrows()):
    axes[0,1].text(i, r['pct_rapido']*100+1, f'{r["pct_rapido"]*100:.1f}%',
                   ha='center', fontweight='bold')

# 3. Distribuição de T: CC* vs CD*, por label
for label, ls, name in [(1,'-','Rápido'), (0,'--','Lento')]:
    for grp, cor in [('CC*','#c0392b'), ('CD*','#2980b9')]:
        mask = (df['grupo'].str.contains(grp[:-1])) & (df['label']==label)
        vals = df[mask]['T']
        if len(vals) > 5:
            axes[1,0].hist(vals, bins=15, alpha=0.4, color=cor,
                          linestyle=ls, histtype='step',
                          linewidth=2, density=True,
                          label=f'{grp} {name}')
axes[1,0].set_xlabel('T (ano entrada em pastagem)')
axes[1,0].set_ylabel('Densidade')
axes[1,0].set_title('Distribuição de T: CC* vs CD*, por label', fontweight='bold')
axes[1,0].legend(fontsize=8)

# 4. Matriz de contingência normalizada
ct = pd.crosstab(df['grupo'], df['label'],
                 normalize='index') * 100
ct.columns = ['Lento (0)', 'Rápido (1)']
im = axes[1,1].imshow(ct.values, cmap='RdYlGn',
                       vmin=0, vmax=100, aspect='auto')
axes[1,1].set_xticks(range(len(ct.columns)))
axes[1,1].set_xticklabels(ct.columns)
axes[1,1].set_yticks(range(len(ct.index)))
axes[1,1].set_yticklabels(
    [g.replace(' (', '\n(') for g in ct.index], fontsize=8)
for i in range(len(ct.index)):
    for j in range(len(ct.columns)):
        axes[1,1].text(j, i, f'{ct.values[i,j]:.1f}%',
                       ha='center', va='center',
                       fontweight='bold', fontsize=10)
plt.colorbar(im, ax=axes[1,1])
axes[1,1].set_title('% por grupo (normalizado por linha)', fontweight='bold')

plt.suptitle('Hipótese: Cluster → Rápido / Disperso → Lento\nDataset v2 (threshold 5 anos)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUT_DIR / 'hypothesis_cluster_rapido.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Veredicto ────────────────────────────────────────────────────────────────
print("=" * 60)
print("VEREDICTO")
print("=" * 60)

diff = pct_cc - pct_cd
sig  = p_cc_cd < 0.05

print(f"\nCC* → {pct_cc:.1f}% rápido")
print(f"CD* → {pct_cd:.1f}% rápido")
print(f"Diferença: {diff:+.1f}pp  (p={p_cc_cd:.4f}, Cohen's h={h:.3f})")
print()

if sig and abs(diff) > 10:
    print("✅ HIPÓTESE CONFIRMADA (efeito grande)")
    print()
    print("   Routing determinístico por padrão é justificado:")
    print("   • CC* → Canal Rápido")
    print("   • CD* + NC → Canal Lento")
    print("   Arquitetura de dois canais pode usar Class8590 diretamente.")
elif sig and abs(diff) > 5:
    print("✅ HIPÓTESE CONFIRMADA (efeito moderado)")
    print()
    print("   Routing determinístico é justificado mas com cautela.")
    print("   A separação existe mas não é perfeita — haverá ruído.")
elif sig:
    print("⚠️  HIPÓTESE PARCIALMENTE CONFIRMADA (efeito pequeno)")
    print()
    print("   Diferença existe mas é pequena.")
    print("   Routing determinístico pode não trazer ganho significativo.")
else:
    print("❌ HIPÓTESE NÃO CONFIRMADA")
    print()
    print("   CC* e CD* têm proporções similares de conversão rápida.")
    print("   Routing por padrão não é justificado pelos dados.")

print(f"\n✅ Gráfico salvo: hypothesis_cluster_rapido.png")